Goals:
Display buildout progress per bid in RDOF.

Current progress: MD, VA, DE

Desciption:
On a national-scale map (more than 1 state),
All census block groups depicted as hexagons.
Display baseline data (June 2022 from NBM) of census block groups served colored dark.
Display planned census block groups served according to RDOF bids colored in a light color that corresponds to the particular RDOF bid.
Display newly served (cumulative) census block groups up to a particular time (July 2022 to December 2025) colored in a dark color.
Color each census block group with saturation corresponding to percentage served.
Every bid should get the same color.


State-scale map:
All RDOF locations depicted as hexagons.
Display baseline data (June 2022 from NBM) of RDOF locations served colored dark.
Display planned RDOF locations served according to RDOF bids colored in a light color that corresponds to the particular RDOF bid (each bid gets its own color)
Display newly served (cumulative) RDOF locations up to a particular time (July 2022 to December 2025) colored in a dark color that corresponds to that particular RDOF bid.


In 2 table below the map,
By CBG or RDOF location:
Show the bid awardee (company),
number of planned CBGs/location,
number of currently built out CBGs/locations (cumulative) that corresponds to the time chosen
number of baseline CBGs/locations, and total number of CBGs/locations.


Above the map, toggle options for which state to show data (select 1-50), toggle options for which companies/bids to show data (1 to all), and a slider from June 2022 to December 2025 in 1 month increments to select the "current time" to show cumulative progress. Also pick view: defaults to CBG for more than one state and RDOF location for 1 state.

In [1]:
# REQUIRED: authenticate with GCP project
GCP_PROJECT_ID = "broadband-data"  #@param {type:"string"}

from google.colab import auth
from google.colab import userdata

auth.authenticate_user(project_id=GCP_PROJECT_ID)

In [2]:
# Authenticate GCP Client
from google.cloud import bigquery

client = bigquery.Client(project=GCP_PROJECT_ID)
print("Authenticated & Output Directory Initialized!")

Authenticated & Output Directory Initialized!


In [3]:
bdc_query = """
WITH rdof_blocks AS (
  SELECT DISTINCT census_block, applicant, LTRIM(frn,'0') AS rdof_frn, sac, county
  FROM `broadband-data.rdof.applicants_hist_2023_cb`
  WHERE state_abbr = 'MD'
),
bids AS (
  SELECT sac, LTRIM(frn,'0') AS rdof_frn,
         SUM(bid_amount) AS bid_amount,
         SUM(n_locations) AS bid_locations,
         STRING_AGG(DISTINCT da_number) AS da_number
  FROM `broadband-data.rdof.applicants_hist_2023`
  WHERE state_abbr = 'MD'
  GROUP BY sac, rdof_frn
),
defaulted AS (
  SELECT DISTINCT census_block
  FROM `broadband-data.rdof.defaults_dedup_cb`
  WHERE state_abbr = 'MD'
),
rdof_locs AS (
  SELECT f.location_id, f.block_geoid, f.county_geoid, f.latitude, f.longitude,
         b.applicant, b.rdof_frn, b.sac,
         bd.bid_amount, bd.bid_locations, bd.da_number,
         d.census_block IS NOT NULL AS block_defaulted
  FROM `broadband-data.bsl_fabric.v7` f
  JOIN rdof_blocks b ON f.block_geoid = b.census_block
  LEFT JOIN bids bd ON bd.sac = b.sac AND bd.rdof_frn = b.rdof_frn
  LEFT JOIN defaulted d ON d.census_block = b.census_block
  WHERE f.bsl_flag AND f.state = 'MD'
)
SELECT
  a.period, a.release,
  l.location_id, l.block_geoid, l.county_geoid, l.latitude, l.longitude,
  l.applicant AS rdof_awardee, l.rdof_frn, l.sac, l.bid_amount, l.bid_locations,
  l.da_number, l.block_defaulted,
  a.frn AS provider_frn, a.provider_id, a.brand_name, a.technology,
  a.max_advertised_download_speed AS down, a.max_advertised_upload_speed AS up,
  a.low_latency, a.business_residential_code,
  LTRIM(a.frn,'0') = l.rdof_frn AS is_awardee
FROM rdof_locs l
LEFT JOIN `broadband-data.fcc_bdc.nbm_hive` a
  ON a.location_id = l.location_id AND a.state_usps = 'MD'
"""

df_bdc = client.query(bdc_query).to_dataframe()
df_bdc.to_parquet("md_rdof_bdc_long.parquet")   # saves in Colab session storage
print(df_bdc.shape, df_bdc.period.nunique(), "periods")

(1945555, 23) 8 periods


In [4]:
"""
Shows how many of each awardee’s RDOF locations it reported serving at 100 Mbps down and 20 Mbps up, in each period.
Shows percentage of the awardee’s total locations.
"""

d = df_bdc.copy()
d["awardee_100_20"] = (d.is_awardee == True) & (d.down >= 100) & (d.up >= 20)
total = d.groupby("rdof_awardee").location_id.nunique().rename("rdof_locations")
prog = (d[d.awardee_100_20].groupby(["rdof_awardee", "period"]).location_id.nunique()
        .unstack("period").fillna(0).join(total))
for c in prog.columns[:-1]:
    prog[c + "_pct"] = (100 * prog[c] / prog["rdof_locations"]).round(1)
prog

,2022_06,2022_12,2023_06,2023_12,2024_06,2024_12,2025_06,2025_12,rdof_locations,2022_06_pct,2022_12_pct,2023_06_pct,2023_12_pct,2024_06_pct,2024_12_pct,2025_06_pct,2025_12_pct
rdof_awardee,,,,,,,,,,,,,,,,,
Easton Utilities Commission,20,70,117,150,207,208,236,239,614,3.3,11.4,19.1,24.4,33.7,33.9,38.4,38.9
"Talkie Communications, Inc.",699,935,1809,2071,2644,3130,3214,4054,37937,1.8,2.5,4.8,5.5,7.0,8.3,8.5,10.7


In [5]:
# ---- Cell 1: one row per RDOF location, for several states ----
# TODO(all states): replace STATES with every state that has RDOF awards (or save each
# state to a BigQuery table and pre-aggregate block groups there, so the national view
# loads a small table instead of millions of locations).
import pandas as pd

STATES = ["MD", "VA", "DE"]
BASELINE = "2022_06"       # baseline = served by the awardee in this NBM filing
SERVED_THRESH = 0.80       # share of locations for a hexagon to count as "served"

def location_query(state):
    return f"""
WITH rdof_blocks AS (
  SELECT DISTINCT census_block, applicant, LTRIM(frn, '0') AS rdof_frn, sac
  FROM `broadband-data.rdof.applicants_hist_2023_cb`
  WHERE state_abbr = '{state}'
    AND census_block NOT IN (
      SELECT census_block FROM `broadband-data.rdof.defaults_dedup_cb`
      WHERE state_abbr = '{state}')
),
rdof_locs AS (
  SELECT f.location_id, DIV(f.block_geoid, 1000) AS cbg, f.latitude, f.longitude, f.h3_9,
         b.applicant AS awardee, b.rdof_frn, b.sac
  FROM `broadband-data.bsl_fabric.v7` f
  JOIN rdof_blocks b ON f.block_geoid = b.census_block
  WHERE f.bsl_flag AND f.state = '{state}'
),
first_served AS (
  SELECT l.location_id, MIN(a.period) AS first_served_period
  FROM rdof_locs l
  JOIN `broadband-data.fcc_bdc.nbm_hive` a
    ON a.location_id = l.location_id
   AND a.state_usps = '{state}'
   AND LTRIM(a.frn, '0') = l.rdof_frn
   -- optional speed filter: AND a.max_advertised_download_speed >= 100
   --                        AND a.max_advertised_upload_speed >= 20
  GROUP BY l.location_id
)
SELECT '{state}' AS state, l.*, fs.first_served_period
FROM rdof_locs l
LEFT JOIN first_served fs USING (location_id)
"""

frames = []
for s in STATES:
    d = client.query(location_query(s)).to_dataframe()
    print(s, f"{len(d):,} locations")
    frames.append(d)
loc = pd.concat(frames, ignore_index=True)
loc["fs_key"] = loc.first_served_period.fillna("9999_99")   # "never served" sorts last

state_list = ", ".join(f"'{s}'" for s in STATES)
periods = sorted(client.query(f"""
    SELECT DISTINCT period FROM `broadband-data.fcc_bdc.nbm_hive`
    WHERE state_usps IN ({state_list})""").to_dataframe().period)

summary = loc.groupby(["state", "awardee"]).agg(
    locations=("location_id", "size"), block_groups=("cbg", "nunique"),
    ever_served=("fs_key", lambda s: int((s <= periods[-1]).sum())))
print("\nAwardees:", loc.awardee.nunique(), "| periods:", periods[0], "to", periods[-1])
summary

MD 38,551 locations
VA 48,678 locations
DE 6,812 locations

Awardees: 18 | periods: 2022_06 to 2025_12


locations  block_groups  \
state awardee                                                                 
DE    Talkie Communications, Inc.                        6812           135   
MD    Easton Utilities Commission                         614             6   
      Talkie Communications, Inc.                       37937           495   
VA    All Points Northern Neck, LLC                      3897            25   
      BARConnects, LLC                                    358             6   
      Central Telephone Company of Virginia              4518            19   
      Central Virginia Services, Inc.                   11729            66   
      Charter Fiberlink VA-CCO, LLC                      5136            28   
      Cogeco US (Delmar), LLC                            1425             6   
      Cox Virginia Telcom, LLC                            241             6   
      EMPOWER Broadband, Inc.                            1755            13   
      FiberLight of Virginia, LLC                         981             6   
      KINEX TELECOM, INC.                                2753            13   
      PGEC Enterprises, LLC                              1813            23   
      Peoples Mutual Telephone Company                    866             7   
      Point Broadband Fiber Holding, LLC                 2153            19   
      RiverStreet Communications of Virginia, Inc.      10017            58   
      Scott County Telephone Cooperative, Inc.            735            15   
      Shenandoah Cable Television, LLC                    301             1   

                                                    ever_served  
state awardee                                                    
DE    Talkie Communications, Inc.                             0  
MD    Easton Utilities Commission                           243  
      Talkie Communications, Inc.                          4666  
VA    All Points Northern Neck, LLC                           0  
      BARConnects, LLC                                        0  
      Central Telephone Company of Virginia                   0  
      Central Virginia Services, Inc.                      7857  
      Charter Fiberlink VA-CCO, LLC                           0  
      Cogeco US (Delmar), LLC                                 0  
      Cox Virginia Telcom, LLC                                0  
      EMPOWER Broadband, Inc.                              1005  
      FiberLight of Virginia, LLC                             0  
      KINEX TELECOM, INC.                                  1552  
      PGEC Enterprises, LLC                                1527  
      Peoples Mutual Telephone Company                      765  
      Point Broadband Fiber Holding, LLC                   1933  
      RiverStreet Communications of Virginia, Inc.         3533  
      Scott County Telephone Cooperative, Inc.                0  
      Shenandoah Cable Television, LLC                        0

In [6]:
# ---- Cell 2: hexagons, status rules, colors ----
!pip -q install "h3>=4"
import h3, colorsys, numpy as np, pandas as pd

LOC_RES = 9   # location view: 9 = about one address per hexagon; use 8 if the map is slow
CBG_RES = 6   # block group view: 5 = big hexagons, 6 = medium, 7 = small

# --- block group view: one distinct hexagon per block group, placed once ---
def place_hexes(info):
    occupied, cells = set(), {}
    for idx, r in info.sort_values("n_total", ascending=False).iterrows():
        home = h3.latlng_to_cell(r.lat, r.lon, CBG_RES)
        cell, k = home, 0
        while cell in occupied:
            k += 1
            free = [c for c in h3.grid_ring(home, k) if c not in occupied]
            if free:
                cell = min(free, key=lambda c: (h3.cell_to_latlng(c)[0] - r.lat) ** 2
                                             + (h3.cell_to_latlng(c)[1] - r.lon) ** 2)
        occupied.add(cell)
        cells[idx] = cell
    return cells

cbg_static = loc.groupby(["awardee", "sac", "cbg"]).agg(
    n_total=("location_id", "size"), lat=("latitude", "mean"), lon=("longitude", "mean"))
CBG_HEX = place_hexes(cbg_static)          # key: (awardee, sac, cbg) -> hexagon id

# --- location view: H3 cell from the Fabric ---
loc["hex_loc"] = [h3.cell_to_parent(c, LOC_RES) for c in loc.h3_9]

_geom = {}
def geom_of(h):
    if h not in _geom:
        ring = [[lng, lat] for lat, lng in h3.cell_to_boundary(h)]   # GeoJSON wants [lng, lat]
        ring.append(ring[0])
        _geom[h] = {"type": "Polygon", "coordinates": [ring]}
    return _geom[h]

# --- one hue per bid, fixed order (palette from the validated categorical set) ---
# TODO(all states): reads well for ~8 bids on screen at once; beyond that, color only the
# selected bids and gray out the rest.
BASE_HEX = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
bids = sorted(loc.awardee.unique())
HUE = {b: colorsys.rgb_to_hls(*[int(BASE_HEX[i % 8][j:j + 2], 16) / 255 for j in (1, 3, 5)])[0]
       for i, b in enumerate(bids)}
LIGHTNESS = {"planned": 0.82, "new": 0.58, "baseline": 0.34}   # light / medium / darkest
LABEL = {"planned": "Planned", "new": "Newly served (cumulative)",
         "baseline": f"Baseline (served by {BASELINE})"}

NATIONAL_HUE = colorsys.rgb_to_hls(*[int("#2a78d6"[j:j + 2], 16) / 255 for j in (1, 3, 5)])[0]

def shade(awardee, cls, pct, single_color=False):
    """Hue = bid (or one shared blue if single_color), lightness = status,
    saturation = share of locations served (20%-100%)."""
    hue = NATIONAL_HUE if single_color else HUE[awardee]
    r, g, b = colorsys.hls_to_rgb(hue, LIGHTNESS[cls], 0.20 + 0.80 * float(pct))
    return "#%02x%02x%02x" % (int(r * 255), int(g * 255), int(b * 255))

def aggregate(df, p, mode):
    """Group locations into hexagons ('locations') or block groups ('block groups')."""
    d = df.assign(sp=df.fs_key <= p, sb=df.fs_key <= BASELINE)
    keys = ["awardee", "sac", "h3_9" if mode == "locations" else "cbg"]
    t = d.groupby(keys).agg(n_total=("location_id", "size"), n_served=("sp", "sum"),
                            n_base=("sb", "sum"), lat=("latitude", "mean"),
                            lon=("longitude", "mean")).reset_index()
    if mode == "locations":
        t["hex"] = [h3.cell_to_parent(c, LOC_RES) for c in t.h3_9]
        t["area"] = t.hex
    else:
        t["hex"] = [CBG_HEX[(r.awardee, r.sac, r.cbg)] for r in t.itertuples()]
        t["area"] = t.cbg.astype(str).str.zfill(12)
    t["pct"] = t.n_served / t.n_total
    t["pct_base"] = t.n_base / t.n_total
    t["cls"] = np.where(t.pct_base >= SERVED_THRESH, "baseline",
                np.where(t.pct >= SERVED_THRESH, "new", "planned"))
    return t

def filing_for(month):
    """Slider month 'YYYY-MM' -> latest NBM filing at or before it ('YYYY_MM')."""
    return [p for p in periods if p.replace("_", "-") <= month][-1]

In [7]:
# ---- Cell 3: map + tables + toggles ----
import folium, ipywidgets as widgets
from IPython.display import display

# If the background tiles are blocked in your notebook, use tiles=None below instead.
TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}"
ATTR = "Tiles © Esri, HERE, Garmin, © OpenStreetMap contributors"
COLS = ["planned", "new", "baseline"]
cbg_state = loc.drop_duplicates("cbg").set_index("cbg").state   # block group -> state

def draw(states_sel, companies_sel, month, view):
    p = filing_for(month)
    sub = loc[loc.state.isin(list(states_sel)) & loc.awardee.isin(list(companies_sel))]
    if sub.empty:
        print("Select at least one state and one company."); return

    # auto: one state -> locations (a color per bid); several states -> block groups (one color)
    mode = ("locations" if len(states_sel) == 1 else "block groups") if view == "Auto" \
           else ("locations" if view == "Locations" else "block groups")
    single = (mode == "block groups")          # one shared color for all bids
    t = aggregate(sub, p, mode)
    t["color"] = [shade(r.awardee, r.cls, r.pct, single) for r in t.itertuples()]
    unit = "RDOF locations in hexagon" if mode == "locations" else "RDOF locations in block group"

    feats = [{"type": "Feature", "geometry": geom_of(r.hex), "properties": {
                "bid": r.awardee, "status": LABEL[r.cls], "pct_served": round(100 * float(r.pct), 1),
                "n_served": int(r.n_served), "n_total": int(r.n_total),
                "area": r.area, "color": r.color}} for r in t.itertuples()]

    m = folium.Map(tiles=TILES, attr=ATTR, prefer_canvas=True)
    m.fit_bounds([[t.lat.min(), t.lon.min()], [t.lat.max(), t.lon.max()]])
    folium.GeoJson(
        {"type": "FeatureCollection", "features": feats},
        style_function=lambda f: {"fillColor": f["properties"]["color"],
                                  "fillOpacity": 0.9, "color": "#ffffff", "weight": 0.4},
        tooltip=folium.GeoJsonTooltip(
            fields=["bid", "status", "pct_served", "n_served", "n_total", "area"],
            aliases=["Bid", "Status", "% served", "Locations served", unit,
                     "Cell" if mode == "locations" else "Block group"]),
    ).add_to(m)

    def swatches(awardee):
        return " ".join(
            f'<span style="color:{shade(awardee, c, 1.0, single)}">&#9632;</span>{LABEL[c].split(" (")[0]}'
            for c in ("baseline", "new", "planned"))

    if not single:   # no legend at the national (block group) level
        legend = "".join(f"<div><b>{b}</b> {swatches(b)}</div>"
                         for b in sorted(sub.awardee.unique()))
        m.get_root().html.add_child(folium.Element(
            '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:#fff;'
            'padding:10px 12px;border:1px solid #ccc;border-radius:6px;font:12px sans-serif;'
            'max-height:40vh;max-width:360px;overflow-y:auto">'
            f'{legend}<div><i>View: {mode}. Saturation = % served. '
            f'Data from the {p} filing (slider: {month}).</i></div></div>'))
    display(m)

    # tables below the map: by company in the location view, by state in the block group view
    cbg = aggregate(sub, p, "block groups")
    cbg["state"] = cbg.cbg.map(cbg_state)
    group = "state" if single else "awardee"
    bg = (cbg.pivot_table(index=group, columns="cls", values="cbg", aggfunc="count", fill_value=0)
             .reindex(columns=COLS, fill_value=0))
    cls_loc = pd.Series(np.select([sub.fs_key <= BASELINE, sub.fs_key <= p], ["baseline", "new"],
                                  default="planned"), index=sub.index, name="cls")
    lc = pd.crosstab(sub[group], cls_loc).reindex(columns=COLS, fill_value=0)
    for tab in (bg, lc):
        tab.columns = ["Planned", "Built since baseline (cumulative)", "Baseline"]
        tab["Total"] = tab.sum(axis=1)
        if single:
            tab.loc["All states"] = tab.sum()
    print("Block groups"); display(bg)
    print("Locations");    display(lc)

months = [str(m) for m in pd.period_range(BASELINE.replace("_", "-"),
                                          periods[-1].replace("_", "-"), freq="M")]
states = sorted(loc.state.unique())
companies = sorted(loc.awardee.unique())
from IPython.display import HTML

from IPython.display import HTML

# ---- state buttons: green + check = shown, gray = not shown ----
# TODO(all states): states fill in automatically once other states are loaded in Cell 1.
state_boxes = {s: widgets.ToggleButton(value=True, description=s, icon="check",
                                       button_style="success", tooltip=f"Show/hide {s}",
                                       layout=widgets.Layout(width="70px"))
               for s in states}
all_btn  = widgets.Button(description="All",  layout=widgets.Layout(width="70px"))
none_btn = widgets.Button(description="None", layout=widgets.Layout(width="70px"))
states_text = widgets.Text(value=",".join(states), layout=widgets.Layout(display="none"))  # hidden carrier

_bulk = False
_syncing = False

def _current_states():
    return ",".join(s for s, b in state_boxes.items() if b.value)

def on_toggle(change):
    b = change["owner"]
    b.button_style = "success" if b.value else ""
    b.icon = "check" if b.value else ""
    if not _bulk:
        states_text.value = _current_states()

def set_all(value):
    global _bulk
    _bulk = True
    for b in state_boxes.values():
        b.value = value
    _bulk = False
    states_text.value = _current_states()

for b in state_boxes.values():
    b.observe(on_toggle, "value")
all_btn.on_click(lambda _: set_all(True))
none_btn.on_click(lambda _: set_all(False))

# ---- other controls ----
company_w = widgets.SelectMultiple(options=companies, value=tuple(companies),
                                   description="Companies", rows=min(max(len(companies), 2), 8),
                                   layout=widgets.Layout(width="420px"))
view_w = widgets.Dropdown(options=["Auto", "Locations", "Block groups"], value="Auto",
                          description="View", layout=widgets.Layout(width="320px"))
month_w = widgets.SelectionSlider(options=months, value=months[-1], description="As of",
                                  layout=widgets.Layout(width="520px"))

def sync_companies(change=None):
    """Companies list = bids that exist in the selected state(s); select all of them."""
    global _syncing
    sel = [s for s in states_text.value.split(",") if s]
    opts = sorted(loc[loc.state.isin(sel)].awardee.unique())
    _syncing = True                    # no redraws while the list is being swapped
    company_w.value = ()               # clear first, so the old selection can't be invalid
    company_w.options = opts
    company_w.rows = min(max(len(opts), 2), 8)
    _syncing = False
    company_w.value = tuple(opts)      # this triggers one redraw

states_text.observe(sync_companies, "value")

# ---- what the map function receives: a clear banner, then the map ----
def banner(sel, n_companies_shown, n_companies_available):
    chip = ('<span style="display:inline-block;margin:0 4px 0 0;padding:2px 10px;border-radius:12px;'
            'font:600 13px sans-serif;{}">{}</span>')
    on  = "".join(chip.format("background:#1f7a3a;color:#fff", s) for s in sel) or "<i>none</i>"
    off = "".join(chip.format("background:#e6e6e6;color:#777;text-decoration:line-through", s)
                  for s in states if s not in sel) or "<i>none</i>"
    display(HTML(f'<div style="font:13px sans-serif;margin:6px 0"><b>Showing:</b> {on} '
                 f'&nbsp;&nbsp; <b>Not showing:</b> {off} &nbsp;&nbsp; '
                 f'<b>Companies:</b> {n_companies_shown} of {n_companies_available}</div>'))

def draw_ui(states_key, companies_sel, month, view):
    if _syncing:
        return
    sel = [s for s in states_key.split(",") if s]
    avail = loc[loc.state.isin(sel)].awardee.nunique()
    banner(sel, len([c for c in companies_sel if c in set(loc[loc.state.isin(sel)].awardee)]), avail)
    if not sel:
        print("No states selected. Click a state button above."); return
    draw(sel, companies_sel, month, view)

out = widgets.interactive_output(
    draw_ui, {"states_key": states_text, "companies_sel": company_w,
              "month": month_w, "view": view_w})
display(widgets.VBox([
    widgets.HBox([widgets.Label("States:"), all_btn, none_btn]),
    widgets.HBox(list(state_boxes.values()), layout=widgets.Layout(flex_flow="row wrap")),
    company_w, view_w, month_w, states_text]), out)
sync_companies()

Output()